In [1]:
import os
import torch
import warnings
warnings.filterwarnings("ignore")
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools.retriever import create_retriever_tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_tavily import TavilySearch
from typing import TypedDict, Annotated, List, Sequence
import operator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHROMA_DB_PATH = "../data/chroma_db"

# Set your Tavily API key
os.environ["TAVILY_API_KEY"] = "Enter_your_api_here"

print(f"Device        : {DEVICE}")
print(f"Torch version : {torch.__version__}")
print("All imports successful ")

Device        : cuda
Torch version : 2.6.0+cu124
All imports successful 


In [2]:
# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True}
)

# Load existing ChromaDB
vectorstore = Chroma(
    persist_directory=CHROMA_DB_PATH,
    embedding_function=embeddings
)
print(f"Chunks in DB  : {vectorstore._collection.count()}")

# LLM
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
    num_ctx=4096,
)
print("LLM loaded")

Chunks in DB  : 211190
LLM loaded


In [3]:
class AgentState(TypedDict):
    messages    : Annotated[List, operator.add]  
    question    : str                       
    context     : str                      
    answer      : str                      
    search_type : str                         
    iterations  : int                          

print("Agent state defined")
print("\nState fields:")
print("  messages    → full conversation history (memory)")
print("  question    → current user question")
print("  context     → retrieved chunks")
print("  answer      → final answer")
print("  search_type → tracks if web search was used")
print("  iterations  → prevents infinite loops")

Agent state defined

State fields:
  messages    → full conversation history (memory)
  question    → current user question
  context     → retrieved chunks
  answer      → final answer
  search_type → tracks if web search was used
  iterations  → prevents infinite loops


In [4]:
from sentence_transformers import CrossEncoder

# Retriever tool
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

retriever_tool = create_retriever_tool(
    retriever,
    name="knowledge_base_search",
    description="""Search the local Wikipedia knowledge base.
    Use this FIRST for any factual question about history, science, 
    geography, technology, people, or general knowledge."""
)

# Web search tool 
web_tool = TavilySearch(
    max_results=3,
    topic="general"
)

# Reranker
reranker = CrossEncoder("BAAI/bge-reranker-base", device="cpu")

def rerank_docs(query: str, docs: list, k: int = 3) -> list:
    if not docs:
        return docs
    pairs  = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:k]]

print("Tools defined:")
print("  1. knowledge_base_search — local Wikipedia (211K chunks)")
print("  2. web_search            — Tavily live web fallback")
print("  3. reranker              — bge-reranker-base on CPU")

Tools defined:
  1. knowledge_base_search — local Wikipedia (211K chunks)
  2. web_search            — Tavily live web fallback
  3. reranker              — bge-reranker-base on CPU


In [5]:
# ── Node 1: Retrieve from local knowledge base ──────────────────
def retrieve_node(state: AgentState) -> AgentState:
    question = state["question"]
    
    # Skip retrieval for meta/memory questions entirely
    meta_keywords = ["previous", "last", "before", "earlier", 
                     "first", "asked", "said", "what did i", 
                     "what was my", "our conversation"]
    
    if any(word in question.lower() for word in meta_keywords):
        print(f"[RETRIEVE] Meta question detected — skipping retrieval")
        return {**state, "context": "", "search_type": "memory"}
    
    print(f"[RETRIEVE] Searching knowledge base for: '{question}'")
    docs = vectorstore.similarity_search(question, k=10)
    docs = rerank_docs(question, docs, k=3)
    context = "\n\n".join([
        f"[Source: {d.metadata['title']}]\n{d.page_content}"
        for d in docs
    ])
    print(f"[RETRIEVE] Found {len(docs)} relevant chunks")
    return {**state, "context": context, "search_type": "local"}


# ── Node 2: Grade retrieval quality ─────────────────────────────
def grade_node(state: AgentState) -> AgentState:
    # Memory questions skip grading
    if state["search_type"] == "memory":
        print(f"[GRADE] Memory question — routing directly to generate")
        return {**state, "search_type": "memory"}
    
    grade_prompt = ChatPromptTemplate.from_template("""
You are a relevance grader. Does this context contain information to answer the question?

Question: {question}
Context (first 800 chars): {context}

Rules:
- Reply 'yes' if context has relevant facts about the question topic
- Reply 'no' if context is unrelated or about a completely different topic
- Reply with ONLY one word: yes or no""")

    chain = grade_prompt | llm | StrOutputParser()
    grade = chain.invoke({
        "question": state["question"],
        "context" : state["context"][:800]
    }).strip().lower()

    is_relevant = "yes" in grade
    print(f"[GRADE] Context relevant: {'yes' if is_relevant else 'no'}")
    return {**state, "search_type": "local" if is_relevant else "web_needed"}

    
# ── Node 3: Web search fallback ──────────────────────────────────
def web_search_node(state: AgentState) -> AgentState:
    """Falls back to Tavily web search if local retrieval is poor"""
    question = state["question"]
    print(f"[WEB SEARCH] Local knowledge insufficient, searching web...")
    
    results  = web_tool.invoke(question)
    
    # Format web results as context
    if isinstance(results, list):
        context = "\n\n".join([
            f"[Web Source: {r.get('url', 'unknown')}]\n{r.get('content', '')}"
            for r in results
        ])
    else:
        context = str(results)
    
    print(f"[WEB SEARCH] Got {len(results) if isinstance(results, list) else 1} results")
    return {**state, "context": context, "search_type": "web"}


# ── Node 4: Generate answer ──────────────────────────────────────
def generate_node(state: AgentState) -> AgentState:
    system_msg = SystemMessage(content="""You are a helpful conversational assistant.
IMPORTANT RULES:
1. Answer ONLY the current question — do not bring up previous topics unprompted
2. If the user asks about conversation history, use ONLY the message history
3. Use the provided context to answer factual questions
4. If context is insufficient, say clearly: "I don't have current information on this"
5. Keep answers focused and concise""")

    question = state["question"]
    history  = state.get("messages", [])

    # Detect meta/memory questions
    meta_keywords = ["previous", "last", "before", "earlier", "first", "asked", "said", "tell me what"]
    is_meta = any(word in question.lower() for word in meta_keywords)

    if is_meta:
        history_text = "\n".join([
            f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}"
            for m in history
        ])
        context_msg = HumanMessage(content=f"""Conversation history:
{history_text}

Current question: {question}
Answer using ONLY the conversation history above. Do not use the retrieved context.""")
        all_msgs = [system_msg, context_msg]

    else:
        context_msg = HumanMessage(content=f"""Retrieved context:
{state['context']}

Current question: {question}
Answer based on the context above. Stay focused on THIS question only.""")
        # Only pass last 4 messages of history to avoid confusion
        all_msgs = [system_msg] + history[-4:] + [context_msg]

    response = llm.invoke(all_msgs)
    answer   = response.content
    print(f"[GENERATE] Answer generated ({len(answer)} chars)")
    print(f"[GENERATE] Source used: {state['search_type']}")
    return {**state, "answer": answer}
    
# ── Node 5: Self-critique ────────────────────────────────────────
def critique_node(state: AgentState) -> AgentState:
    """Checks if answer is grounded and complete"""
    critique_prompt = ChatPromptTemplate.from_template("""
Check if this answer is grounded in the context and complete.

Question: {question}
Context: {context}
Answer: {answer}

Reply with ONLY one word: 'good' if answer is grounded, 'retry' if not.""")
    
    chain  = critique_prompt | llm | StrOutputParser()
    result = chain.invoke({
        "question": state["question"],
        "context" : state["context"][:500],
        "answer"  : state["answer"][:500]
    }).strip().lower()
    
    print(f"[CRITIQUE] Answer quality: {result}")
    iterations = state.get("iterations", 0) + 1
    return {**state, "iterations": iterations}


print("All nodes defined")
print("\nAgent nodes:")
print("  1. retrieve  → search local ChromaDB + rerank")
print("  2. grade     → is context relevant?")
print("  3. web_search→ Tavily fallback if grade=no")
print("  4. generate  → LLM answer with memory")
print("  5. critique  → is answer grounded?")

All nodes defined

Agent nodes:
  1. retrieve  → search local ChromaDB + rerank
  2. grade     → is context relevant?
  3. web_search→ Tavily fallback if grade=no
  4. generate  → LLM answer with memory
  5. critique  → is answer grounded?


In [8]:
# ── Routing functions ────────────────────────────────────────────
def route_after_grade(state: AgentState) -> str:
    if state["search_type"] == "memory":
        return "generate"
    if state["search_type"] == "web_needed":
        return "web_search"
    return "generate"

def route_after_critique(state: AgentState) -> str:
    """Don't retry — just return what we have. Retrying causes confusion."""
    if state["iterations"] >= 1:
        print("[ROUTE] Returning best answer found")
        return END
    # Only retry if answer is completely empty
    if not state.get("answer", "").strip():
        return "web_search"
    return END
    
# ── Build graph ──────────────────────────────────────────────────
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("retrieve",   retrieve_node)
graph.add_node("grade",      grade_node)
graph.add_node("web_search", web_search_node)
graph.add_node("generate",   generate_node)
graph.add_node("critique",   critique_node)

# Entry point
graph.set_entry_point("retrieve")

# Edges
graph.add_edge("retrieve",   "grade")
graph.add_conditional_edges("grade", route_after_grade, {
    "web_search": "web_search",
    "generate"  : "generate"
})
graph.add_edge("web_search", "generate")
graph.add_edge("generate",   "critique")
graph.add_conditional_edges("critique", route_after_critique, {
    "web_search": "web_search",
    END         : END
})

# Compile with memory checkpointer
memory = MemorySaver()
agent  = graph.compile(checkpointer=memory)

print("LangGraph agent compiled")
print("\nGraph flow:")
print("  retrieve → grade → [web_search]? → generate → critique → [retry]? → END")

LangGraph agent compiled

Graph flow:
  retrieve → grade → [web_search]? → generate → critique → [retry]? → END


In [9]:
def chat(question: str, thread_id: str = "default") -> str:
    """
    Single function to chat with the agent.
    thread_id keeps separate conversation histories.
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    # Build state
    state = {
        "messages"   : [HumanMessage(content=question)],
        "question"   : question,
        "context"    : "",
        "answer"     : "",
        "search_type": "local",
        "iterations" : 0
    }
    
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    
    # Run agent
    result = agent.invoke(state, config=config)
    answer = result["answer"]
    
    print(f"\nA: {answer}")
    print(f"\n[Used: {result['search_type']} search]")
    return None

# Test single question
chat("What is hall effect?", thread_id="session_01")


Q: What is hall effect?
[RETRIEVE] Searching knowledge base for: 'What is hall effect?'
[RETRIEVE] Found 3 relevant chunks
[GRADE] Context relevant: yes
[GENERATE] Answer generated (286 chars)
[GENERATE] Source used: local
[CRITIQUE] Answer quality: good
[ROUTE] Returning best answer found

A: The Hall effect is a phenomenon where a voltage develops across conductors transverse to an electric current in the conductor and magnetic field perpendicular to the current, arising due to the nature of charge carriers in the conductor. It was discovered by Edwin Herbert Hall in 1879.

[Used: local search]


In [10]:
# This tests memory — follow-up questions should use previous context

print("\n" + "="*60)
print("CONVERSATIONAL MEMORY TEST")
print("="*60)

# Turn 1
chat("Tell me about Albert Einstein.", thread_id="conv_test")

# Turn 2 — follow-up using "he" — tests if agent remembers Einstein
chat("Where was he born?", thread_id="conv_test")

# Turn 3 — another follow-up
chat("What is he most famous for?", thread_id="conv_test")


CONVERSATIONAL MEMORY TEST

Q: Tell me about Albert Einstein.
[RETRIEVE] Searching knowledge base for: 'Tell me about Albert Einstein.'
[RETRIEVE] Found 3 relevant chunks
[GRADE] Context relevant: yes
[GENERATE] Answer generated (308 chars)
[GENERATE] Source used: local
[CRITIQUE] Answer quality: good
[ROUTE] Returning best answer found

A: Albert Einstein was a German-born theoretical physicist who is widely considered one of the greatest scientists of all time. He is best known for developing the theory of relativity and making important contributions to quantum mechanics, which revolutionized our understanding of nature in the 20th century.

[Used: local search]

Q: Where was he born?
[RETRIEVE] Searching knowledge base for: 'Where was he born?'
[RETRIEVE] Found 3 relevant chunks
[GRADE] Context relevant: yes
[GENERATE] Answer generated (242 chars)
[GENERATE] Source used: local
[CRITIQUE] Answer quality: retry
[ROUTE] Returning best answer found

A: I don't have current information

In [11]:
# Test with a recent/specific question that's NOT in Wikipedia 2023 dump
print("\n" + "="*60)
print("WEB FALLBACK TEST")
print("="*60)

# This should trigger web search since it's very recent news
chat("What are the latest AI models released in 2025?", thread_id="web_test")


WEB FALLBACK TEST

Q: What are the latest AI models released in 2025?
[RETRIEVE] Searching knowledge base for: 'What are the latest AI models released in 2025?'
[RETRIEVE] Found 3 relevant chunks
[GRADE] Context relevant: no
[WEB SEARCH] Local knowledge insufficient, searching web...
[WEB SEARCH] Got 1 results
[GENERATE] Answer generated (537 chars)
[GENERATE] Source used: web
[CRITIQUE] Answer quality: good
[ROUTE] Returning best answer found

A: According to the provided context, some of the latest AI models released in 2025 include:

* OpenAI GPT-5 / GPT-5.1
* Google Gemini 3
* Anthropic Claude 4 (Opus & Sonnet)
* Meta Llama 4 (Scout)

Additionally, other notable releases mentioned in the context are:
* Alibaba's model that claims to outperform GPT-4o, DeepSeek-V3, and Meta's Llama 3.1.
* Nano Banana's image editing and visual capabilities.

Please note that this is not an exhaustive list, as the provided context only includes a few examples of AI models released in 2025.

[Used: w

In [12]:
# Full interactive chat — type your questions!
print("="*60)
print("AGENTIC RAG CHATBOT — Interactive Mode")
print("Type 'quit' to exit, 'new' for new conversation")
print("="*60)

session_id  = "interactive_01"
turn_count  = 0

while True:
    user_input = input("\nYou: ").strip()
    
    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("Goodbye!")
        break
    if user_input.lower() == "new":
        session_id = f"interactive_{turn_count}"
        print(f"Started new conversation (session: {session_id})")
        continue
    
    turn_count += 1
    chat(user_input, thread_id=session_id)

AGENTIC RAG CHATBOT — Interactive Mode
Type 'quit' to exit, 'new' for new conversation



You:  what is qunatum tunneling?



Q: what is qunatum tunneling?
[RETRIEVE] Searching knowledge base for: 'what is qunatum tunneling?'
[RETRIEVE] Found 3 relevant chunks
[GRADE] Context relevant: yes
[GENERATE] Answer generated (481 chars)
[GENERATE] Source used: local
[CRITIQUE] Answer quality: good
[ROUTE] Returning best answer found

A: Quantum tunneling is a phenomenon in which particles can pass through a barrier that would be insurmountable in the classical perspective, allowing them to escape or move through spaces they shouldn't be able to reach according to classical physics. It's an exponential process dependent on the separation between two points, and it's observed in various physical systems, such as electrons tunneling through a vacuum between electrodes, or alpha particles escaping from a nucleus.

[Used: local search]



You:  what is the current condition of strait of hormuz?



Q: what is the current condition of strait of hormuz?
[RETRIEVE] Searching knowledge base for: 'what is the current condition of strait of hormuz?'
[RETRIEVE] Found 3 relevant chunks
[GRADE] Context relevant: no
[WEB SEARCH] Local knowledge insufficient, searching web...
[WEB SEARCH] Got 1 results
[GENERATE] Answer generated (376 chars)
[GENERATE] Source used: web
[CRITIQUE] Answer quality: retry
[ROUTE] Returning best answer found

A: According to the provided information, Iran's foreign minister has stated that the Strait of Hormuz is now fully open to commercial vessels (source: The Guardian). However, I don't have real-time information on the current condition of the strait. It was mentioned in the context that there were overcast clouds at some point, but this might not reflect the current situation.

[Used: web search]


...


Q: what was the first question i asked you?
[RETRIEVE] Meta question detected — skipping retrieval
[GRADE] Memory question — routing directly to generate
[GENERATE] Answer generated (78 chars)
[GENERATE] Source used: memory
[CRITIQUE] Answer quality: retry
[ROUTE] Returning best answer found

A: The first question you asked me was "what was the first question I asked you?"

[Used: memory search]



You:  what is the first prompt i gave you in this session



Q: what is the first prompt i gave you in this session
[RETRIEVE] Meta question detected — skipping retrieval
[GRADE] Memory question — routing directly to generate
[GENERATE] Answer generated (77 chars)
[GENERATE] Source used: memory
[CRITIQUE] Answer quality: retry
[ROUTE] Returning best answer found

A: The first prompt you gave me in this session was "what is qunatum tunneling?"

[Used: memory search]



You:  quit


Goodbye!
